# Equipo 19 · Prueba de SAM para las máscaras de reflejos

**Pregunta:** ¿SAM (*Segment Anything*, Kirillov et al., 2023) detecta los reflejos especulares mejor que El
Meslouhi y Arnold? Es solo el **primer paso**: medir las máscaras. Si SAM no gana claramente, no vale la pena
reentrenar Endo-STTN con ellas.

SAM no sabe qué es un reflejo: segmenta lo que se le señala. Por eso se prueban tres formas de usarlo:

| Método | Cómo funciona |
|---|---|
| `meslouhi`, `arnold` | Los métodos actuales, sin SAM (línea base) |
| `sam_meslouhi` | **SAM como refinador:** cada reflejo detectado por El Meslouhi se le da a SAM como caja + punto, y SAM ajusta su contorno |
| `sam_arnold` | Lo mismo, partiendo de Arnold |
| `sam_auto` | **SAM automático:** segmenta toda la imagen y se quedan los segmentos pequeños, muy brillantes y casi blancos |

Todas se miden contra la **misma referencia física** que en el Avance 3. Los videos registrados de C3VDv2 traen
`diffuse/`, un render del modelo 3D sin componente especular; un píxel mucho más brillante de lo que predice ese
render es un reflejo. Se reportan precisión, sensibilidad, F1 e IoU (estrictos y con tolerancia de 2 px en los
bordes), separando videos limpios (v1, v2) de videos con residuos (v3). También se mide el **tiempo por frame**, que
importa para decidir si es viable usar SAM en miles de frames.

## 0. Antes de empezar

**Colab Pro:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución →* **A100 GPU** (o L4). Después:
*Entorno de ejecución → Ejecutar todas*.

| En Google Drive | Qué es |
|---|---|
| `MyDrive/Equipo19/raw/*.zip` | ZIPs de C3VDv2. Solo entran los que traen `diffuse/` (videos registrados) |

Los pesos de SAM (`sam_vit_h_4b8939.pth`, 2.4 GB) se descargan solos la primera vez y se guardan en
`MyDrive/Equipo19/SAM_prueba/`. Los resultados quedan en esa misma carpeta.

In [ ]:
from pathlib import Path
import os, sys, re, time, zipfile, subprocess, urllib.request

# ---------- Ajusta aquí ----------
FRAMES_POR_VIDEO = 30        # frames equiespaciados por video (los mismos que en el Avance 3)
MODELO_SAM = "vit_h"         # "vit_h" (el más preciso), "vit_l" o "vit_b" (el más rápido)
CORRER_SAM_AUTO = True       # el modo automático es el más lento; ponlo en False para saltarlo

# Referencia de diffuse/ (los mismos valores que en el Avance 3)
K, Y_MIN, MEDIANA, TOLERANCIA = 1.3, 0.6, 31, 2

# SAM como refinador
MARGEN = 3                   # px que se agranda la caja de cada reflejo detectado
MIN_AREA = 4                 # componentes más chicos se dejan como están (SAM no es fiable con 1-3 px)
MAX_CRECIMIENTO = 4.0        # si SAM devuelve algo más de 4 veces más grande que el reflejo detectado,
                             # tomó tejido: se conserva la detección original
# SAM automático
PUNTOS_AUTO = 32             # rejilla de puntos por lado (más puntos = encuentra reflejos más chicos, pero más lento)
AREA_MAX_AUTO = 0.01         # un reflejo ocupa menos del 1 % del frame
V_MIN_AUTO, S_MAX_AUTO = 200, 60   # brillo medio mínimo y saturación media máxima (HSV, 0-255)

MYDRIVE = Path("/content/drive/MyDrive")
CARPETA_RAW = MYDRIVE / "Equipo19" / "raw"
SALIDA = MYDRIVE / "Equipo19" / "SAM_prueba"
# ---------------------------------

assert "google.colab" in sys.modules, "Este notebook es para Google Colab"
from google.colab import drive
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segment-anything", "tabulate"], check=True)

SALIDA.mkdir(parents=True, exist_ok=True)
PESOS_SAM = {"vit_h": "sam_vit_h_4b8939.pth", "vit_l": "sam_vit_l_0b3195.pth", "vit_b": "sam_vit_b_01ec64.pth"}
CHECKPOINT = SALIDA / PESOS_SAM[MODELO_SAM]
if not CHECKPOINT.exists():
    print("Descargando", CHECKPOINT.name, "(solo la primera vez)...")
    urllib.request.urlretrieve(f"https://dl.fbaipublicfiles.com/segment_anything/{CHECKPOINT.name}", f"{CHECKPOINT}.tmp")
    os.replace(f"{CHECKPOINT}.tmp", CHECKPOINT)
print("SAM:", MODELO_SAM, CHECKPOINT)

## 1. Herramientas

**1a. Lectura de C3VDv2, referencia de `diffuse/` y métricas.** Es la misma lógica de
`Avance3_ReEntrenar_Endo/comparar_mascaras.py`, copiada aquí para que el notebook no dependa del repositorio. Todo se
mide a **288×288**, la resolución con la que trabaja Endo-STTN: si SAM se adopta, sus máscaras tendrían que estar a
esa resolución.

Referencia, con $Y$ el brillo del frame real y $D$ la imagen difusa (0–1):

$$s = \text{mediana local}(Y/D), \qquad \text{reflejo} \iff Y \ge Y_{min} \;\wedge\; Y > K \cdot s \cdot D$$

In [ ]:
import cv2
import numpy as np
import pandas as pd

LADO = 288


def indice(nombre):
    m = re.search(r"(\d+)", Path(nombre).stem)
    return int(m.group(1)) if m else None


def pares_difusa(zf):
    """(índice, ruta rgb, ruta diffuse) de los frames que tienen las dos imágenes."""
    nombres = zf.namelist()
    rgb = {indice(n): n for n in nombres if n.startswith("rgb/") and n.lower().endswith((".png", ".jpg"))}
    dif = {indice(n): n for n in nombres if n.startswith("diffuse/") and n.lower().endswith((".png", ".tif", ".tiff"))}
    return [(i, rgb[i], dif[i]) for i in sorted(set(rgb) & set(dif) - {None})]


def frame_288(zf, ruta):
    """Frame de color a 288x288, pasando por 320x256 como en el resto del proyecto."""
    rgb = cv2.cvtColor(cv2.imdecode(np.frombuffer(zf.read(ruta), np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (320, 256), interpolation=cv2.INTER_AREA)
    return cv2.resize(rgb, (LADO, LADO), interpolation=cv2.INTER_AREA)


def difusa_288(zf, ruta):
    """Imagen difusa en gris, 0-1, a 288x288 (acepta 8 o 16 bits, 1 o 3 canales)."""
    d = cv2.imdecode(np.frombuffer(zf.read(ruta), np.uint8), cv2.IMREAD_UNCHANGED)
    maximo = float(np.iinfo(d.dtype).max) if np.issubdtype(d.dtype, np.integer) else max(float(d.max()), 1.0)
    d = d.astype(np.float32) / maximo
    if d.ndim == 3:
        d = d[..., :3].mean(-1)
    d = cv2.resize(d, (320, 256), interpolation=cv2.INTER_AREA)
    return cv2.resize(d, (LADO, LADO), interpolation=cv2.INTER_AREA)


def referencia(rgb, difusa):
    """Reflejo = mucho más brillante de lo que predice el render difuso."""
    y = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255
    valido = difusa > 0.02
    cociente = np.where(valido, y / np.maximum(difusa, 0.02), 0)
    escala = cv2.medianBlur(np.clip(cociente * 50, 0, 255).astype(np.uint8), MEDIANA).astype(np.float32) / 50
    return valido & (y >= Y_MIN) & (y > K * escala * difusa)


def comparar(pred, ref):
    """Conteos estrictos y con tolerancia de TOLERANCIA px (para sumarlos entre frames)."""
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * TOLERANCIA + 1, 2 * TOLERANCIA + 1))
    dil = lambda m: cv2.dilate(m.astype(np.uint8), ker).astype(bool)
    return {"tp": int((pred & ref).sum()), "fp": int((pred & ~ref).sum()), "fn": int((~pred & ref).sum()),
            "pred_ok_tol": int((pred & dil(ref)).sum()), "ref_ok_tol": int((ref & dil(pred)).sum()),
            "pred": int(pred.sum()), "ref": int(ref.sum()), "pixeles": int(pred.size)}


def resumir(df):
    """Precisión, sensibilidad, F1 e IoU sumando los píxeles de todos los frames del grupo."""
    s = df.sum(numeric_only=True)
    div = lambda a, b: a / b if b else float("nan")
    p, r = div(s.tp, s.tp + s.fp), div(s.tp, s.tp + s.fn)
    pt, rt = div(s.pred_ok_tol, s.pred), div(s.ref_ok_tol, s.ref)
    f1 = lambda a, b: div(2 * a * b, a + b)
    return pd.Series({"precision": p, "sensibilidad": r, "f1": f1(p, r), "iou": div(s.tp, s.tp + s.fp + s.fn),
                      "precision_tol": pt, "sensibilidad_tol": rt, "f1_tol": f1(pt, rt),
                      "marcado_pct": 100 * div(s.pred, s.pixeles), "referencia_pct": 100 * div(s.ref, s.pixeles),
                      "frames": len(df)})


VIDEOS = {}
for z in sorted(CARPETA_RAW.glob("*.zip")):
    with zipfile.ZipFile(z) as zf:
        if any(n.startswith("diffuse/") for n in zf.namelist()):
            VIDEOS[z.stem] = z
print(f"Videos con diffuse/ en {CARPETA_RAW}: {len(VIDEOS)}")
for v in VIDEOS:
    print("  ", v)
assert VIDEOS, "Ningún ZIP trae diffuse/: hace falta al menos un video registrado"

**1b. Las máscaras actuales** (El Meslouhi y Arnold), idénticas a las que se usaron para reentrenar Endo-STTN en el
Avance 4. Se calculan sobre el frame a 288×288 y se dilatan 1 px.

In [ ]:
def _srgb_a_xyz(img):
    lin = np.where(img <= 0.04045, img / 12.92, ((img + 0.055) / 1.055) ** 2.4)
    m = np.array([[0.4124564, 0.3575761, 0.1804375],
                  [0.2126729, 0.7151522, 0.0721750],
                  [0.0193339, 0.1191920, 0.9503041]])
    return lin @ m.T


def dilatar_1px(m):
    return cv2.dilate(m.astype(np.uint8), np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.uint8)).astype(bool)


def mascara_meslouhi(rgb):
    """El Meslouhi et al. (2011): realce por min/max, CIE-XYZ, reflejo donde Y >= Y/(X+Y+Z)."""
    img = rgb.astype(np.float64) / 255
    mn, mx = img.min(-1), img.max(-1)
    s = np.divide(mn, mx, out=np.zeros_like(mn), where=mx > 0)
    xyz = _srgb_a_xyz(s[..., None] * img)
    suma = xyz.sum(-1)
    y = np.divide(xyz[..., 1], suma, out=np.full_like(suma, np.inf), where=suma > 0)
    return dilatar_1px(xyz[..., 1] >= y)


def mascara_arnold(rgb, t_abs=240, t_cand=200, t_rel=1.2, mediana=31):
    """Arnold et al. (2010), versión simplificada: detector absoluto + detector relativo."""
    img = rgb.astype(np.float32)
    gris = img @ np.array([0.2989, 0.5870, 0.1140], np.float32)
    p95 = lambda c: max(float(np.percentile(c, 95)), 1.0)
    r_g, r_b = p95(img[..., 1]) / p95(gris), p95(img[..., 2]) / p95(gris)
    sobre = lambda t: (img[..., 1] > r_g * t) | (img[..., 2] > r_b * t) | (gris > t)
    candidatos = cv2.dilate(sobre(t_cand).astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
    sin_reflejos = cv2.medianBlur(cv2.inpaint(rgb, candidatos, 5, cv2.INPAINT_TELEA), mediana).astype(np.float32)
    relativo = ((img / (sin_reflejos + 1)).max(-1) > t_rel) & (gris > t_cand * 0.75)
    return dilatar_1px(sobre(t_abs) | relativo)

**1c. SAM.** Las dos formas de usarlo:

- **Refinador** (`refinar`): la detección actual se separa en componentes (cada reflejo aislado). Por cada uno, SAM
  recibe una **caja** alrededor del reflejo (agrandada `MARGEN` px) y un **punto positivo** en su píxel más
  brillante, y devuelve el contorno. Hay tres salvaguardas: el resultado se recorta a la caja, los componentes muy
  pequeños se dejan como estaban, y si SAM devuelve algo mucho más grande que el reflejo (`MAX_CRECIMIENTO`) se
  considera que tomó tejido y se conserva la detección original. Al final se dilata 1 px, igual que las otras.
- **Automático** (`sam_automatico`): SAM segmenta todo el frame con una rejilla de puntos y se conservan los
  segmentos pequeños (menos de `AREA_MAX_AUTO` del frame), muy brillantes (`V_MIN_AUTO`) y casi blancos
  (`S_MAX_AUTO`): las tres propiedades de un reflejo según el modelo dicromático.

SAM trabaja internamente a 1024 px, así que el frame de 288×288 se amplía; eso ayuda a que los reflejos pequeños
ocupen más píxeles.

In [ ]:
import torch
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry[MODELO_SAM](checkpoint=str(CHECKPOINT)).to(DEVICE).eval()
predictor = SamPredictor(sam)
generador = SamAutomaticMaskGenerator(sam, points_per_side=PUNTOS_AUTO, pred_iou_thresh=0.8,
                                      stability_score_thresh=0.85)
print("Dispositivo:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu (sin GPU: será muy lento)")


@torch.no_grad()
def refinar(rgb, semilla, lote=64):
    """SAM ajusta el contorno de cada reflejo detectado. Requiere predictor.set_image(rgb) antes.

    Devuelve la máscara y cuántos componentes se refinaron y cuántos se rechazaron (SAM tomó tejido)."""
    gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    n, etiquetas, stats, _ = cv2.connectedComponentsWithStats(semilla.astype(np.uint8), connectivity=8)
    out = np.zeros(semilla.shape, bool)
    cajas, puntos, ids = [], [], []
    for k in range(1, n):
        x, y, w, h, area = stats[k]
        if area < MIN_AREA:
            out |= etiquetas == k
            continue
        ys, xs = np.nonzero(etiquetas == k)
        j = int(np.argmax(gris[ys, xs]))
        cajas.append([max(x - MARGEN, 0), max(y - MARGEN, 0), min(x + w - 1 + MARGEN, LADO - 1), min(y + h - 1 + MARGEN, LADO - 1)])
        puntos.append([xs[j], ys[j]])
        ids.append(k)
    rechazados = 0
    for a in range(0, len(ids), lote):
        c = torch.tensor(cajas[a:a + lote], dtype=torch.float, device=DEVICE)
        p = torch.tensor(puntos[a:a + lote], dtype=torch.float, device=DEVICE)[:, None, :]
        masks, _, _ = predictor.predict_torch(
            point_coords=predictor.transform.apply_coords_torch(p, rgb.shape[:2]),
            point_labels=torch.ones(len(p), 1, device=DEVICE),
            boxes=predictor.transform.apply_boxes_torch(c, rgb.shape[:2]), multimask_output=False)
        for m, (x0, y0, x1, y1), k in zip(masks[:, 0].cpu().numpy(), cajas[a:a + lote], ids[a:a + lote]):
            dentro = np.zeros_like(m)
            dentro[y0:y1 + 1, x0:x1 + 1] = m[y0:y1 + 1, x0:x1 + 1]
            if dentro.sum() == 0 or dentro.sum() > MAX_CRECIMIENTO * stats[k, cv2.CC_STAT_AREA]:
                out |= etiquetas == k
                rechazados += 1
            else:
                out |= dentro
    return dilatar_1px(out), {"componentes": len(ids), "rechazados": rechazados}


@torch.no_grad()
def sam_automatico(rgb):
    """Segmentos de SAM que son pequeños, muy brillantes y casi blancos."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    out = np.zeros(rgb.shape[:2], bool)
    for s in generador.generate(rgb):
        m = s["segmentation"]
        if m.mean() <= AREA_MAX_AUTO and hsv[..., 2][m].mean() >= V_MIN_AUTO and hsv[..., 1][m].mean() <= S_MAX_AUTO:
            out |= m
    return dilatar_1px(out)


def todas_las_mascaras(rgb):
    """Máscara de cada método y segundos que tardó cada uno en este frame."""
    mascaras, segundos, info = {}, {}, {}
    for nombre, f in [("meslouhi", mascara_meslouhi), ("arnold", mascara_arnold)]:
        t0 = time.time(); mascaras[nombre] = f(rgb); segundos[nombre] = time.time() - t0
    t0 = time.time(); predictor.set_image(rgb); t_imagen = time.time() - t0   # una vez por frame, la usan los dos
    for base in ("meslouhi", "arnold"):
        t0 = time.time()
        mascaras[f"sam_{base}"], info[f"sam_{base}"] = refinar(rgb, mascaras[base])
        segundos[f"sam_{base}"] = segundos[base] + t_imagen + time.time() - t0
    if CORRER_SAM_AUTO:
        t0 = time.time(); mascaras["sam_auto"] = sam_automatico(rgb); segundos["sam_auto"] = time.time() - t0
    return mascaras, segundos, info

## 2. Revisar la referencia

Antes de confiar en los números hay que ver que la referencia sí marque reflejos. En el Avance 3 la referencia marcó
alrededor del 0.6 % de los píxeles, mucho menos que cualquier método (4–8 %), y todos los F1 salieron bajos
(0.15–0.20 con tolerancia). Por eso aquí interesa sobre todo la **comparación relativa** entre métodos, no el valor
absoluto.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

MAGENTA = np.array([255, 0, 255])
sobre = lambda img, m: np.where(m[..., None], (0.3 * img + 0.7 * MAGENTA).astype(np.uint8), img)


def muestra(ruta, n):
    """n frames equiespaciados de un video: lista de (índice, rgb 288, difusa 288)."""
    with zipfile.ZipFile(ruta) as zf:
        pares = pares_difusa(zf)
        elegidos = [pares[i] for i in np.linspace(0, len(pares) - 1, min(n, len(pares))).astype(int)]
        return [(i, frame_288(zf, r), difusa_288(zf, d)) for i, r, d in elegidos]


fig, ax = plt.subplots(len(VIDEOS), 3, figsize=(11, 3.6 * len(VIDEOS)), squeeze=False)
for fila, (v, ruta) in zip(ax, VIDEOS.items()):
    i, rgb, dif = muestra(ruta, 3)[1]
    ref = referencia(rgb, dif)
    for a, (img, t, cmap) in zip(fila, [(rgb, f"{v} · {i}", None), (dif, "diffuse/ (render sin reflejos)", "gray"),
                                        (sobre(rgb, ref), f"referencia ({100 * ref.mean():.2f} %)", None)]):
        a.imshow(img, cmap=cmap); a.set_title(t, fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

## 3. Medir los cinco métodos

Por cada frame se calculan todas las máscaras, se comparan contra la referencia y se guarda cuánto tardó cada
método. Con `vit_h` en una A100, el modo automático es el que más tarda.

In [ ]:
filas, tiempos, rechazos = [], [], []
for v, ruta in VIDEOS.items():
    version = (re.search(r"_v(\d+)", v) or [None, "?"])[1]
    t0 = time.time()
    for i, rgb, dif in muestra(ruta, FRAMES_POR_VIDEO):
        ref = referencia(rgb, dif)
        mascaras, segundos, info = todas_las_mascaras(rgb)
        for metodo, pred in mascaras.items():
            filas.append({"video": v, "version": f"v{version}", "frame": i, "metodo": metodo, **comparar(pred, ref)})
            tiempos.append({"metodo": metodo, "segundos": segundos[metodo]})
        for metodo, d in info.items():
            rechazos.append({"metodo": metodo, **d})
    print(f"{v}: {FRAMES_POR_VIDEO} frames en {time.time() - t0:.0f} s", flush=True)

df = pd.DataFrame(filas)
grupos = {"limpios (v1, v2)": df[df.version.isin(["v1", "v2"])], "con residuos (v3)": df[df.version == "v3"], "todos": df}
res = pd.DataFrame([{"videos": g, "metodo": m, **resumir(dm)} for g, d in grupos.items() for m, dm in d.groupby("metodo")])
METODOS = [m for m in ["meslouhi", "arnold", "sam_meslouhi", "sam_arnold", "sam_auto"] if m in set(df.metodo)]
costo = pd.DataFrame(tiempos).groupby("metodo").segundos.mean().rename("segundos_por_frame")
rech = pd.DataFrame(rechazos).groupby("metodo").sum()
rech["rechazados_pct"] = 100 * rech.rechazados / rech.componentes.clip(lower=1)

df.to_csv(SALIDA / "comparacion_por_frame.csv", index=False)
res.to_csv(SALIDA / "comparacion_resumen.csv", index=False)
cols = ["precision", "sensibilidad", "f1", "f1_tol", "iou", "marcado_pct", "referencia_pct", "frames"]
for g in grupos:
    if res[res.videos == g].frames.sum():
        display(Markdown(f"**{g}**"))
        display(res[res.videos == g].set_index("metodo").loc[METODOS, cols].round(3))
display(Markdown("**Costo y refinamiento**"))
display(pd.concat([costo, rech[["componentes", "rechazados_pct"]]], axis=1).loc[METODOS].round(3))

## 4. Gráficas

Izquierda: F1 con tolerancia por grupo de videos. Derecha: precisión contra sensibilidad en los videos limpios. Un
buen refinador debería **subir la precisión** (marcar menos tejido) sin perder mucha sensibilidad respecto a su
método de partida; en la gráfica, una flecha de cada método base a su versión con SAM muestra ese cambio.

In [ ]:
COLOR = {"meslouhi": "#2a78d6", "arnold": "#1baf7a", "sam_meslouhi": "#6250d6", "sam_arnold": "#eda100", "sam_auto": "#eb6834"}
NOMBRE = {"meslouhi": "El Meslouhi", "arnold": "Arnold", "sam_meslouhi": "SAM + El Meslouhi",
          "sam_arnold": "SAM + Arnold", "sam_auto": "SAM automático"}
con_datos = [g for g in grupos if res[res.videos == g].frames.sum()]
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ancho = 0.8 / len(METODOS)
for j, m in enumerate(METODOS):
    vals = [res[(res.videos == g) & (res.metodo == m)].f1_tol.iloc[0] for g in con_datos]
    xs = np.arange(len(con_datos)) + (j - (len(METODOS) - 1) / 2) * ancho
    ax[0].bar(xs, vals, ancho * 0.9, color=COLOR[m], label=NOMBRE[m])
    for x, val in zip(xs, vals):
        ax[0].text(x, val + 0.005, f"{val:.2f}", ha="center", fontsize=8, color="#52514e")
ax[0].set_xticks(range(len(con_datos))); ax[0].set_xticklabels(con_datos)
ax[0].set_ylabel("F1 con tolerancia de 2 px"); ax[0].set_title("Acuerdo con la referencia (mayor es mejor)")
ax[0].legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1), frameon=False); ax[0].grid(axis="y", alpha=0.25)

base = res[res.videos == con_datos[0]].set_index("metodo")
for m in METODOS:
    ax[1].plot(base.sensibilidad_tol[m], base.precision_tol[m], "o", ms=11, color=COLOR[m], label=NOMBRE[m])
for b in ("meslouhi", "arnold"):
    if f"sam_{b}" in base.index:
        ax[1].annotate("", (base.sensibilidad_tol[f"sam_{b}"], base.precision_tol[f"sam_{b}"]),
                       (base.sensibilidad_tol[b], base.precision_tol[b]), arrowprops={"arrowstyle": "->", "color": "#8a8985"})
ax[1].set_xlim(0, 1.02); ax[1].set_ylim(0, max(0.3, base.precision_tol.max() * 1.2))
ax[1].set_xlabel("sensibilidad (encuentra los reflejos)"); ax[1].set_ylabel("precisión (no marca otra cosa)")
ax[1].set_title(f"Precisión vs sensibilidad · {con_datos[0]}"); ax[1].legend(fontsize=9); ax[1].grid(alpha=0.25)
plt.tight_layout(); plt.savefig(SALIDA / "comparacion_metricas.png", dpi=110, bbox_inches="tight"); plt.show()

## 5. Comparación visual

Para un frame de cada video, con zoom donde hay reflejos: **verde** = acierto, **naranja** = marcado de más (falso
positivo), **azul** = reflejo no marcado (falso negativo). Los números no bastan: aquí se ve si SAM ajusta mejor los
bordes o si "se escapa" hacia el tejido.

In [ ]:
TP, FP, FN = np.array([27, 175, 122]), np.array([235, 104, 52]), np.array([42, 120, 214])


def zoom_en(m, lado=128):
    ys, xs = np.nonzero(m) if m.any() else (np.array([144]), np.array([144]))
    y0 = int(np.clip(np.median(ys) - lado // 2, 0, LADO - lado)); x0 = int(np.clip(np.median(xs) - lado // 2, 0, LADO - lado))
    return lambda a: a[y0:y0 + lado, x0:x0 + lado]


fig, ax = plt.subplots(len(VIDEOS), len(METODOS) + 2, figsize=(3.3 * (len(METODOS) + 2), 3.5 * len(VIDEOS)), squeeze=False)
for fila, (v, ruta) in zip(ax, VIDEOS.items()):
    i, rgb, dif = muestra(ruta, 3)[1]
    ref = referencia(rgb, dif)
    mascaras, _, _ = todas_las_mascaras(rgb)
    z = zoom_en(ref | mascaras["meslouhi"])
    fila[0].imshow(z(rgb)); fila[0].set_title(f"{v} · {i} (zoom)", fontsize=9)
    fila[1].imshow(z(sobre(rgb, ref))); fila[1].set_title("referencia (magenta)", fontsize=9)
    for a, m in zip(fila[2:], METODOS):
        p, r = z(mascaras[m]), z(ref)
        img = z(rgb).astype(float)
        for sel, c in [(p & r, TP), (p & ~r, FP), (~p & r, FN)]:
            img[sel] = 0.25 * img[sel] + 0.75 * c
        f1 = resumir(pd.DataFrame([comparar(mascaras[m], ref)])).f1_tol
        a.imshow(img.astype(np.uint8)); a.set_title(f"{NOMBRE[m]} · F1 tol {f1:.2f}", fontsize=9)
for a in ax.ravel():
    a.axis("off")
plt.suptitle("verde = acierto · naranja = de más · azul = no marcado", y=1.0)
plt.tight_layout(); plt.savefig(SALIDA / "comparacion_visual.png", dpi=90, bbox_inches="tight"); plt.show()

## 6. ¿Vale la pena seguir con SAM?

Regla para decidir (la misma idea del Avance 3): se toma el F1 con tolerancia en los videos **limpios** y se compara
el mejor método con SAM contra el mejor método actual. Solo una ventaja **clara** (más de 0.02) justifica el
siguiente paso, que es reentrenar Endo-STTN con las máscaras de SAM. También pesa el costo: si SAM tarda segundos
por frame, preparar los miles de frames del reentrenamiento será mucho más lento.

In [ ]:
b = res[res.videos == con_datos[0]].set_index("metodo")
actuales = [m for m in METODOS if not m.startswith("sam")]
con_sam = [m for m in METODOS if m.startswith("sam")]
mejor_actual, mejor_sam = b.f1_tol[actuales].idxmax(), b.f1_tol[con_sam].idxmax()
ventaja = b.f1_tol[mejor_sam] - b.f1_tol[mejor_actual]
texto = (f"Grupo: {con_datos[0]}\n"
         f"Mejor método actual: {NOMBRE[mejor_actual]} (F1 tol {b.f1_tol[mejor_actual]:.3f}, {costo[mejor_actual]:.2f} s/frame)\n"
         f"Mejor método con SAM: {NOMBRE[mejor_sam]} (F1 tol {b.f1_tol[mejor_sam]:.3f}, {costo[mejor_sam]:.2f} s/frame)\n"
         f"Ventaja de SAM: {ventaja:+.3f}\n\n")
texto += ("SAM gana claramente: vale la pena reentrenar Endo-STTN con sus máscaras (paso 2)." if ventaja > 0.02 else
          "SAM no gana claramente: no conviene reentrenar Endo-STTN con sus máscaras; quedarse con El Meslouhi o Arnold.")
print(texto)
with open(SALIDA / "comparacion_resumen.md", "w") as f:
    f.write("# SAM contra El Meslouhi y Arnold (referencia de diffuse/)\n\n")
    f.write(f"SAM {MODELO_SAM}. Referencia: Y >= {Y_MIN} y Y > {K} x predicción difusa. {FRAMES_POR_VIDEO} frames por "
            f"video, {len(VIDEOS)} videos registrados. Tolerancia en bordes: {TOLERANCIA} px.\n\n")
    f.write(res.round(3).to_markdown(index=False) + "\n\n")
    f.write(pd.concat([costo, rech[["componentes", "rechazados_pct"]]], axis=1).loc[METODOS].round(3).to_markdown() + "\n\n")
    f.write("```\n" + texto + "\n```\n")
print("\nResultados en", SALIDA)

## 7. Cómo interpretarlo

- **SAM + (El Meslouhi o Arnold) sube la precisión sin perder sensibilidad:** SAM ajusta mejor los bordes y deja de
  marcar tejido alrededor. Es el resultado que justificaría usarlo como refinador.
- **Baja la sensibilidad:** SAM "encoge" los reflejos o los descarta. Revisa la comparación visual: los reflejos con
  halo difuso suelen perderse.
- **Muchos rechazados** (columna `rechazados_pct`): SAM tiende a tomar tejido alrededor del reflejo. Prueba
  `MARGEN = 1` o un `MAX_CRECIMIENTO` menor.
- **SAM automático muy por debajo:** es lo esperado. SAM no sabe qué es un reflejo y la rejilla de puntos no alcanza
  los reflejos más pequeños. Se puede subir `PUNTOS_AUTO` a 64, a cambio de unas cuatro veces más tiempo.
- **Todos los F1 son bajos:** la referencia de `diffuse/` no es perfecta (depende del registro 3D y de `K`, `Y_MIN`).
  Lo que importa es la comparación entre métodos con la misma referencia.